# Visualization Overview

Consolidated notebook covering **every visualization type** not already present in the other five notebooks in this folder.

| Already covered elsewhere | This notebook adds |
|---|---|
| Missing-value heatmap (`eda`) | Class-conditional violin / box plots |
| Class balance bar (`eda`) | Feature × feature scatter (top pairs) |
| Feature histograms — class overlay (`eda`) | Per-ticker feature time series |
| Spearman / Pearson heatmaps (`statistical_analysis`) | Model feature-importance side-by-side |
| Dendrogram + VIF (`statistical_analysis`) | Calibration curves |
| ACF / PACF / Ljung-Box (`correlation_extension`) | Confusion matrices (per fold) |
| Rolling & lagged correlation (`correlation_extension`) | Q-Q plots (predicted probabilities) |
| Temporal cyclic features (`temporal_data_analysis`) | t-SNE feature space |
| TIRP / KLS patterns (`temporal_analysis`) | Cross-ticker GapUp rate comparison |
| Walk-forward AUC folds (`temporal_extensions`) | Annual regime summary radar |

**Data:** `dataset_clean_temporal.csv` — primary set (`is_extreme_event == 0`), 25 tickers, 2016–2024.

## 1  Imports & Data

In [ ]:
from __future__ import annotations
import warnings, pickle
warnings.filterwarnings('ignore')

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy.stats import spearmanr, probplot
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix, roc_auc_score

plt.rcParams.update({'figure.dpi':120,'font.size':10,
                     'axes.titlesize':11,'figure.facecolor':'white'})
BLUE='#4c8af5'; RED='#f55c5c'; GREY='#aaaaaa'
DIV='RdBu_r'; SEQ='YlOrRd'

ROOT  = Path('..').resolve()
TPATH = ROOT/'structured_csv_data_files'/'fetched_data'/'dataset_clean_temporal.csv'

df_all  = pd.read_csv(TPATH)
df_all['Date'] = pd.to_datetime(df_all['Date'], utc=True).dt.tz_convert(None)
primary = df_all[df_all['is_extreme_event']==0].copy()
primary = primary.sort_values(['Ticker','Date']).reset_index(drop=True)

TARGET = 'GapUp'
MOMENTUM   = ['RSI','MACD','ROC','StochPercK','CloseVEma50','CloseVSma20']
VOLATILITY = ['ADX','BollingerBandWidth','ATR','FiveDStdDev','IntraWeekVolatility','WeeklyRange']
VOLUME     = ['OBV','MFI','VolumeRatio']
PRICE_MICRO= ['WeeklyReturn','FridayPosition','OpenCloseSpread','GapMagnitude']
FUNDAMENTAL= ['NetMargin','RoA','RevGrowthQoQ','GrossMargin']
TEMPORAL_F = ['Month_sin','Month_cos','WeekOfYear_sin','WeekOfYear_cos',
               'Quarter_sin','Quarter_cos','DaysSinceStart']
ALL_FEATURES = MOMENTUM+VOLATILITY+VOLUME+PRICE_MICRO+FUNDAMENTAL+TEMPORAL_F
BASELINE     = ['MACD','ROC','StochPercK','CloseVEma50','CloseVSma20','ADX',
                'BollingerBandWidth','ATR','FiveDStdDev','OBV','MFI','VolumeRatio',
                'NetMargin','RoA','RevGrowthQoQ']
GROUP_MAP = {f:g for g,fl in [
    ('Momentum',MOMENTUM),('Volatility',VOLATILITY),('Volume',VOLUME),
    ('PriceMicro',PRICE_MICRO),('Fundamental',FUNDAMENTAL),('Temporal',TEMPORAL_F)
] for f in fl}
GROUP_COLORS = {
    'Momentum':'#4c8af5','Volatility':'#f5a623','Volume':'#7ed321',
    'PriceMicro':'#9b59b6','Fundamental':'#e74c3c','Temporal':'#1abc9c',
}
print(f'Rows: {len(primary):,}  Tickers: {primary.Ticker.nunique()}  GapUp rate: {primary[TARGET].mean():.3f}')

## 2  Class-Conditional Violin + Box Plots

For every feature in every group, side-by-side violin plots split by `GapUp = 0` vs `GapUp = 1`.  
Wider violins indicate more density; inner box = IQR; white dot = median.

**Interpretation:** Features where the two violins barely overlap are strong class separators.  
Features with nearly identical violin shapes carry little discriminative information.

> EDA notebooks show class-overlay histograms; violins simultaneously convey shape, spread, and median shift in a single compact panel.

In [ ]:
feature_groups = [
    ('Momentum',   MOMENTUM),
    ('Volatility', VOLATILITY),
    ('Volume',     VOLUME),
    ('PriceMicro', PRICE_MICRO),
    ('Fundamental',FUNDAMENTAL),
    ('Temporal',   TEMPORAL_F),
]

for group_name, feats in feature_groups:
    ncols = 3
    nrows = int(np.ceil(len(feats)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4*nrows))
    axes = np.array(axes).flatten()
    color = GROUP_COLORS[group_name]

    for ax, feat in zip(axes, feats):
        data0 = primary[primary[TARGET]==0][feat].dropna()
        data1 = primary[primary[TARGET]==1][feat].dropna()
        # clip to 1-99th pctile to reduce outlier stretch
        lo, hi = np.percentile(primary[feat].dropna(), [1,99])
        d0 = data0.clip(lo, hi); d1 = data1.clip(lo, hi)
        parts = ax.violinplot([d0, d1], positions=[0,1],
                              showmedians=True, showextrema=False)
        for pc in parts['bodies']:
            pc.set_facecolor(color); pc.set_alpha(0.55)
        parts['cmedians'].set_color('black'); parts['cmedians'].set_lw(2)
        ax.boxplot([d0, d1], positions=[0,1], widths=0.08,
                   patch_artist=True,
                   boxprops=dict(facecolor='white', alpha=0.7),
                   medianprops=dict(color='black'),
                   whiskerprops=dict(alpha=0),
                   capprops=dict(alpha=0),
                   flierprops=dict(alpha=0))
        ax.set_xticks([0,1]); ax.set_xticklabels(['GapUp=0','GapUp=1'], fontsize=8)
        ax.set_title(feat, fontsize=9)
        ax.set_ylabel('Value (clipped 1-99%)', fontsize=7)

    for ax in axes[len(feats):]: ax.axis('off')
    fig.suptitle(f'Class-Conditional Violin Plots — {group_name} Features',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

## 3  Feature × Feature Scatter — Top Correlated Pairs (coloured by GapUp)

Scatter plots of the 9 most collinear feature pairs (|Spearman ρ| > 0.70).  
Points are coloured by `GapUp` — blue = no gap-up, red = gap-up.

**Purpose:** Identifies whether high collinearity still contains discriminative structure.  
If both classes blend uniformly along the collinear axis, one of the two features is redundant.  
If one class clusters in a corner, the pair together is more powerful than either alone.

In [ ]:
from scipy.stats import spearmanr as spr

HIGH_PAIRS = [(f1,f2) for f1,f2 in combinations(ALL_FEATURES,2)
              if abs(spr(primary[f1].values, primary[f2].values, nan_policy='omit')[0]) > 0.70]

ncols = 3
nrows = int(np.ceil(len(HIGH_PAIRS)/ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.5*nrows))
axes = np.array(axes).flatten()

sample = primary.sample(min(3000, len(primary)), random_state=42)
for ax, (f1,f2) in zip(axes, HIGH_PAIRS):
    rho,_ = spr(sample[f1].values, sample[f2].values, nan_policy='omit')
    c = [BLUE if v==0 else RED for v in sample[TARGET]]
    ax.scatter(sample[f1], sample[f2], c=c, alpha=0.25, s=8, rasterized=True)
    ax.set_xlabel(f1, fontsize=8); ax.set_ylabel(f2, fontsize=8)
    ax.set_title(f'rho={rho:+.3f}', fontsize=9)

for ax in axes[len(HIGH_PAIRS):]: ax.axis('off')
legend_h = [mpatches.Patch(color=BLUE,label='GapUp=0'),
            mpatches.Patch(color=RED, label='GapUp=1')]
fig.legend(handles=legend_h, loc='lower right', fontsize=9)
fig.suptitle('Top Collinear Feature Pairs — Scatter coloured by GapUp
'
             '(|Spearman rho| > 0.70; 3 000-row sample)', fontsize=12)
plt.tight_layout()
plt.show()

## 4  Per-Ticker Feature Time Series

Rolling 26-week median of the top-4 features (by |Spearman ρ with GapUp|) for every ticker.  
GapUp events are overlaid as orange markers.

**Purpose:** Reveals whether individual tickers show idiosyncratic feature regimes vs.  
sector-wide synchronisation.  High cross-ticker similarity → pooled model is appropriate.  
Divergent trajectories → ticker-specific calibration or stratified modelling needed.

In [ ]:
# top 4 features by |Spearman rho| — computed quickly
top4 = sorted(ALL_FEATURES,
              key=lambda f: abs(spr(primary[f].values, primary[TARGET].values,
                                   nan_policy='omit')[0]),
              reverse=True)[:4]

tickers = sorted(primary['Ticker'].unique())
fig, axes = plt.subplots(len(top4), 1, figsize=(16, 12), sharex=False)

for ax, feat in zip(axes, top4):
    grp_color = GROUP_COLORS[GROUP_MAP[feat]]
    for ticker in tickers:
        td = primary[primary['Ticker']==ticker].sort_values('Date')
        roll = td[feat].rolling(26, min_periods=5).median()
        ax.plot(td['Date'], roll, lw=0.7, alpha=0.45, color=grp_color)
    # cross-sectional median
    weekly = primary.groupby('Date')[feat].median()
    ax.plot(weekly.index, weekly.rolling(26,min_periods=5).mean(),
            color='black', lw=2.0, label='Cross-sectional median (26-wk)')
    # GapUp events (cross-sectional median date)
    gap_dates = primary[primary[TARGET]==1]['Date'].unique()
    ax.scatter(gap_dates,
               [weekly.reindex(gap_dates).median()]*len(gap_dates),
               color='orange', s=6, alpha=0.4, zorder=5, label='GapUp events')
    ax.set_ylabel(feat, fontsize=9)
    ax.legend(fontsize=7, loc='upper left')

axes[0].set_title('Per-Ticker 26-Week Rolling Median — Top-4 Features
'
                  '(faint lines = individual tickers; bold = cross-sectional median)')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

## 5  Feature Importance — LogReg Coefficients vs XGBoost Gain

Side-by-side bar charts comparing how each model ranks the baseline features.

- **LogReg:** absolute value of the standardised coefficient (direction preserved by colour).
- **XGBoost:** mean gain importance (average improvement in loss brought by each split on that feature).

Features that rank highly in *both* models are the most reliable predictors.  
Divergence between models reveals which features are useful only for linear vs. non-linear decision boundaries.

In [ ]:
import pickle, warnings; warnings.filterwarnings('ignore')

with open(ROOT/'final_outputs'/'logreg_best.pkl','rb') as f: lr_pkl = pickle.load(f)
with open(ROOT/'final_outputs'/'xgboost_best.pkl','rb') as f: xgb_pkl = pickle.load(f)

lr_model   = lr_pkl['model']
xgb_model  = xgb_pkl['model']
xgb_feats  = xgb_pkl['features']

# LR coefficients (use the preprocessing pipeline if available)
if hasattr(lr_model, 'coef_'):
    lr_coef = lr_model.coef_[0]
    # If PCA was applied, coefficients are in PC space — skip LR detail
    lr_pca  = lr_pkl.get('pca', False)
else:
    lr_coef = None; lr_pca = True

# XGB gain importance
xgb_imp = xgb_model.get_booster().get_score(importance_type='gain')
xgb_imp_df = (pd.Series(xgb_imp)
               .reindex(xgb_feats).fillna(0)
               .sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# XGB
colors_xgb = [GROUP_COLORS[GROUP_MAP.get(f,'Momentum')] for f in xgb_imp_df.index]
axes[0].barh(xgb_imp_df.index, xgb_imp_df.values, color=colors_xgb, edgecolor='white')
axes[0].invert_yaxis()
axes[0].set_title('XGBoost Feature Importance (Mean Gain)')
axes[0].set_xlabel('Mean Gain')

# LR
if lr_pca:
    axes[1].text(0.5, 0.5,
                 'LR was trained in PCA space\n—\ncoefficients map to PCs,\nnot original features.',
                 ha='center', va='center', fontsize=12, transform=axes[1].transAxes)
    axes[1].set_title('LogReg Coefficients (PCA variant — not directly comparable)')
else:
    lr_df = pd.Series(lr_coef, index=xgb_feats[:len(lr_coef)]).sort_values(key=abs, ascending=False)
    colors_lr = ['#e74c3c' if v>0 else '#3498db' for v in lr_df.values]
    axes[1].barh(lr_df.index, lr_df.values, color=colors_lr, edgecolor='white')
    axes[1].axvline(0, color='black', lw=0.8)
    axes[1].invert_yaxis()
    axes[1].set_title('LogReg Coefficients (red=positive, blue=negative)')
    axes[1].set_xlabel('Coefficient')

handles = [mpatches.Patch(color=c,label=g) for g,c in GROUP_COLORS.items()]
fig.legend(handles=handles, loc='lower right', fontsize=8, ncol=3)
fig.suptitle('Model Feature Importance Comparison — LogReg vs XGBoost', fontsize=12)
plt.tight_layout()
plt.show()

## 6  Calibration Curves — Predicted Probability vs Actual Frequency

A well-calibrated model produces predicted probabilities that match empirical class frequencies.  
The **diagonal** = perfect calibration.

- Curve **above** diagonal → model under-predicts (over-confident in the negative class).
- Curve **below** diagonal → model over-predicts positive probability.

We derive predicted probabilities using the saved models by re-scoring the primary dataset  
(train-on-all approximation — not a proper hold-out, but useful for calibration shape diagnosis).

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler

# Re-score the full primary set for calibration shape
X = primary[xgb_feats].fillna(0).values
y = primary[TARGET].values

# XGB probabilities
xgb_proba = xgb_model.predict_proba(X)[:,1]

fig, ax = plt.subplots(figsize=(8,6))
ax.plot([0,1],[0,1], 'k--', lw=1, label='Perfect calibration')

for n_bins, ls in [(10,'solid'),(15,'dashed')]:
    frac_pos, mean_pred = calibration_curve(y, xgb_proba, n_bins=n_bins)
    ax.plot(mean_pred, frac_pos, marker='o', markersize=4,
            linestyle=ls, label=f'XGBoost ({n_bins} bins)', color=BLUE)

ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives (actual GapUp rate)')
ax.set_title('Calibration Curve — XGBoost\n'
             '(full primary set re-scored; useful for shape, not hold-out AUC)')
ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.show()

# Histogram of predicted probabilities
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(xgb_proba[y==0], bins=40, alpha=0.6, color=BLUE,  label='GapUp=0', density=True)
ax.hist(xgb_proba[y==1], bins=40, alpha=0.6, color=RED,   label='GapUp=1', density=True)
ax.set_xlabel('Predicted probability of GapUp')
ax.set_ylabel('Density')
ax.set_title('Distribution of Predicted Probabilities by True Class — XGBoost')
ax.legend()
plt.tight_layout()
plt.show()

## 7  Confusion Matrices — Per-Year Breakdown

Confusion matrix for XGBoost predictions on the full primary set, shown for each calendar year.  
This reveals whether the model's error type (false positives vs false negatives) is regime-dependent.

- **FP spike** in a year → model predicts too many gap-ups (possibly correlated with high volatility regimes).
- **FN spike** → model misses actual gap-ups (possibly during unusual market conditions).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

years = sorted(primary['Year'].unique())
ncols = 3
nrows = int(np.ceil(len(years)/ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.5*nrows))
axes = np.array(axes).flatten()

threshold = 0.5
for ax, yr in zip(axes, years):
    mask = primary['Year']==yr
    y_t  = primary.loc[mask, TARGET].values
    p_t  = xgb_proba[mask]
    y_p  = (p_t >= threshold).astype(int)
    cm   = confusion_matrix(y_t, y_p)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Gap','Gap Up'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    auc  = roc_auc_score(y_t, p_t) if y_t.std()>0 else float('nan')
    ax.set_title(f'{yr}  (AUC={auc:.3f})', fontsize=9)

for ax in axes[len(years):]: ax.axis('off')
fig.suptitle('XGBoost Confusion Matrices by Year  (threshold = 0.50)', fontsize=12)
plt.tight_layout()
plt.show()

## 8  Q-Q Plot — Predicted Probability Distribution

A Q-Q plot comparing the distribution of XGBoost predicted probabilities to a uniform distribution.

- If probabilities were **perfectly uniform** (random model), points would follow the diagonal.
- Deviation at the tails shows the model's tendency to concentrate predictions near 0 or 1.
- **S-curve** = bimodal distribution (model is confident in both directions).
- **Convex** = model hedges toward the centre (under-confident).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Q-Q vs uniform
(osm, osr), (slope, intercept, r) = probplot(xgb_proba, dist='uniform', fit=True)
axes[0].scatter(osm, osr, s=4, alpha=0.3, color=BLUE, rasterized=True)
line_x = np.array([osm.min(), osm.max()])
axes[0].plot(line_x, slope*line_x + intercept, 'r-', lw=1.5)
axes[0].set_xlabel('Theoretical quantiles (Uniform[0,1])')
axes[0].set_ylabel('Predicted probability quantiles')
axes[0].set_title(f'Q-Q Plot: XGBoost Proba vs Uniform  (R²={r**2:.3f})')

# Sorted probability plot (rank plot)
sorted_p = np.sort(xgb_proba)
axes[1].plot(np.linspace(0,1,len(sorted_p)), sorted_p, color=BLUE, lw=1.2)
axes[1].plot([0,1],[0,1],'k--',lw=0.8,label='Uniform reference')
axes[1].fill_between(np.linspace(0,1,len(sorted_p)), sorted_p,
                     np.linspace(0,1,len(sorted_p)), alpha=0.15, color=BLUE)
axes[1].set_xlabel('Cumulative rank')
axes[1].set_ylabel('Predicted probability')
axes[1].set_title('Sorted Probability Plot — XGBoost')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9  t-SNE — Feature Space 2-D Projection

t-SNE reduces the 15-feature baseline space to 2 dimensions, preserving local neighbourhood structure.  
Points are coloured by `GapUp`.

**If** the two classes form distinct clusters → the feature space is linearly or non-linearly separable.  
**If** classes are fully interleaved → the boundary is highly irregular and model generalisation will be hard.

> t-SNE is stochastic; `random_state=42` is fixed for reproducibility.  
> Uses a 2 000-row stratified sample to keep runtime under 30 seconds.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

sample_idx = (primary.groupby(TARGET, group_keys=False)
              .apply(lambda g: g.sample(min(1000,len(g)), random_state=42))
              .index)
Xs = primary.loc[sample_idx, BASELINE].fillna(0).values
ys = primary.loc[sample_idx, TARGET].values

Xs_sc = StandardScaler().fit_transform(Xs)
embed = TSNE(n_components=2, perplexity=30, random_state=42,
             n_iter=1000, verbose=0).fit_transform(Xs_sc)

fig, ax = plt.subplots(figsize=(10,7))
for cls, color, label in [(0,BLUE,'GapUp=0'),(1,RED,'GapUp=1')]:
    mask = ys==cls
    ax.scatter(embed[mask,0], embed[mask,1],
               c=color, alpha=0.5, s=12, label=label, rasterized=True)

ax.set_title('t-SNE — Baseline Feature Space (2 000-row sample)
'
             'Coloured by GapUp target')
ax.legend(fontsize=9)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

## 10  Cross-Ticker GapUp Rate Comparison

Bar chart of GapUp rate per ticker, overall and split by year.

High variance across tickers suggests ticker-specific modelling could improve performance.  
Low variance confirms the pooled-model assumption is reasonable.

In [ ]:
ticker_rate = (primary.groupby('Ticker')[TARGET].mean()
               .sort_values(ascending=False))
yr_rate     = (primary.groupby(['Ticker','Year'])[TARGET].mean()
               .unstack(fill_value=np.nan))

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Overall rate
colors = plt.cm.RdYlGn([v for v in ticker_rate.values])
axes[0].bar(ticker_rate.index, ticker_rate.values, color=colors, edgecolor='white')
axes[0].axhline(primary[TARGET].mean(), color='black', lw=1.2, linestyle='--',
                label=f'Overall mean = {primary[TARGET].mean():.3f}')
axes[0].set_ylabel('GapUp rate')
axes[0].set_title('GapUp Rate per Ticker (all years)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# Year × ticker heatmap
sns.heatmap(yr_rate.T, ax=axes[1], cmap='RdYlGn',
            center=primary[TARGET].mean(),
            annot=True, fmt='.2f', annot_kws={'size':7},
            linewidths=0.4, linecolor='white',
            cbar_kws={'label':'GapUp rate'})
axes[1].set_title('GapUp Rate — Year × Ticker Heatmap')
axes[1].set_xlabel('Ticker'); axes[1].set_ylabel('Year')

plt.tight_layout()
plt.show()

## 11  Annual Regime Radar Chart

A radar (spider) chart showing 6 normalised market-regime indicators per calendar year.  
Reveals which years are structurally similar and which are anomalous (e.g. COVID 2020).

Indicators: mean GapUp rate, mean GapMagnitude, mean RSI, mean ATR, mean BollingerBandWidth, mean VolumeRatio.

In [ ]:
RADAR_FEATS = ['GapMagnitude','RSI','ATR','BollingerBandWidth','VolumeRatio','WeeklyReturn']
year_means  = primary.groupby('Year')[RADAR_FEATS+[TARGET]].mean()

# normalise 0-1
norm = (year_means - year_means.min()) / (year_means.max() - year_means.min() + 1e-9)
norm['GapUp'] = year_means[TARGET]

angles = np.linspace(0, 2*np.pi, len(RADAR_FEATS), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9,9), subplot_kw=dict(polar=True))
cmap = plt.cm.tab10
years_list = norm.index.tolist()

for k, yr in enumerate(years_list):
    vals = norm.loc[yr, RADAR_FEATS].tolist()
    vals += vals[:1]
    lw  = 2.5 if yr==2020 else 1.4
    ls  = '--' if yr==2020 else '-'
    ax.plot(angles, vals, lw=lw, linestyle=ls,
            color=cmap(k/len(years_list)), label=str(yr))
    ax.fill(angles, vals, alpha=0.05, color=cmap(k/len(years_list)))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_FEATS, fontsize=9)
ax.set_title('Annual Regime Radar — Normalised Feature Means\n'
             '(dashed = COVID 2020)', fontsize=11, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout()
plt.show()

## 12  Pair Plot — Top 5 Features by |Spearman ρ|

Seaborn pair plot of the 5 features most correlated with GapUp, coloured by target.  
Diagonal = KDE per class.  Off-diagonal = scatter.

Useful for spotting non-linear structure that individual Spearman ρ values miss.

In [ ]:
top5 = sorted(ALL_FEATURES,
              key=lambda f: abs(spr(primary[f].values, primary[TARGET].values,
                                   nan_policy='omit')[0]),
              reverse=True)[:5]
top5 = [f for f in top5 if f != 'GapMagnitude'][:5]  # exclude near-perfect leakage proxy

sample2 = primary.sample(min(2000,len(primary)), random_state=0)
pp_df   = sample2[top5+[TARGET]].dropna()
pp_df[TARGET] = pp_df[TARGET].map({0:'GapUp=0',1:'GapUp=1'})

g = sns.pairplot(pp_df, hue=TARGET,
                 palette={'GapUp=0':BLUE,'GapUp=1':RED},
                 plot_kws=dict(alpha=0.25, s=8, rasterized=True),
                 diag_kind='kde', diag_kws=dict(alpha=0.5),
                 corner=True)
g.figure.suptitle('Pair Plot — Top 5 Features by |Spearman rho| with GapUp\n'
                  '(2 000-row sample; GapMagnitude excluded as leakage proxy)',
                  y=1.01, fontsize=11)
plt.tight_layout()
plt.show()